# 코랩 A100 · 수혁 AUG 모델(0.941) TTA + 동현 TTA 모델과 앙상블

1. 수혁 님 본선 모델(8B · 640px · 증강 · 리더보드 0.941)로 test 를 **다른 해상도 · 보기 순서로 다시 풀기** (재학습 없음)
2. 원래 640px 결과와 평균 → **수혁 TTA 모델**
3. 드라이브의 **동현 TTA 모델(0.9568)** 을 찾아 자동으로 섞어서 **제출 파일 여러 개** 생성

**남은 41유닛 기준**: 768px + 보기 2칸 이동 두 조합 (약 1.4시간 · 로드 포함 약 33유닛). 속도를 실제로 재서 예산 안에 드는 것만 돌립니다.

### 준비
- 드라이브에 `수혁_AUG_epoch.zip` (epoch2 폴더 통째로) · `data.zip`
- 런타임 **A100** · 모두 실행 · 설치 셀 뒤 import 에러가 나면 **세션 다시 시작 → 모두 실행**
- 조합마다 드라이브에 저장 → 끊겨도 다시 실행하면 이어서 진행

In [ ]:
CONFIG = {
    "RUN_NAME":     "수혁_AUG",
    "BASE_REPO":    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",   # 수혁 PC 학습과 같은 4bit 기본 모델
    "LOAD_IN_4BIT": True,
    "BUDGET_HOURS": 1.4,
    "PASSES": [(768, 0), (640, 2), (896, 0)],   # 위에서부터 예산 안에 드는 만큼. (640, 0) 은 이미 있음
}
print(CONFIG["RUN_NAME"], "| 예산", CONFIG["BUDGET_HOURS"], "시간")

# 1. 드라이브 · 어댑터 · 데이터

In [ ]:
import os, subprocess, zipfile, unicodedata
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
N = lambda s: unicodedata.normalize("NFC", s)
MY = Path("/content/drive/MyDrive")

# --- 어댑터 zip (한글 이름 인코딩 차이도 처리) ---
zips = [p for p in MY.rglob("*.zip") if "수혁_AUG" in N(p.name)]
assert zips, "드라이브에서 '수혁_AUG...zip' 을 찾지 못했습니다"
dest = Path("/content/adapter"); dest.mkdir(exist_ok=True)
if not list(dest.rglob("adapter_config.json")):
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(dest)
ADAPTER_DIR = list(dest.rglob("adapter_config.json"))[0].parent
names = sorted(f.name for f in ADAPTER_DIR.iterdir())
assert "adapter_model.safetensors" in names and "test_probs.npy" in names, names
print("어댑터:", ADAPTER_DIR, names)

# --- 데이터 ---
dz = next(MY.rglob("data.zip"), None); assert dz, "data.zip 없음"
WORK = Path("/content/challenge"); WORK.mkdir(exist_ok=True)
if not list(WORK.rglob("test.csv")):
    subprocess.run(["unzip", "-q", str(dz), "-d", str(WORK)], check=True)
DATA_DIR = next(WORK.rglob("test.csv")).parent

# --- 결과는 드라이브에 (세션이 끊겨도 남음) ---
OUT = MY / "challenge" / "output"
TTA_DIR = OUT / "tta_수혁_AUG"; TTA_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = OUT / "submissions"; SUB_DIR.mkdir(parents=True, exist_ok=True)
print("데이터:", DATA_DIR, "| TTA 저장:", TTA_DIR)

# 2. 설치

In [ ]:
!pip install -q unsloth unsloth_zoo

# 3. 불러오기 · A100 확인

In [ ]:
from unsloth import FastVisionModel
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
gpu = torch.cuda.get_device_name(); vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | {vram:.0f}GB")
assert "A100" in gpu or vram > 38, "A100 이 아닙니다. 런타임 유형을 A100 으로 바꾸고 다시 연결하세요"

# 4. test 데이터 · 수혁 640px 확률

In [ ]:
import numpy as np, pandas as pd, time
from PIL import Image
from tqdm.auto import tqdm
Image.MAX_IMAGE_PIXELS = None
test_df    = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
def image_path(rel):
    return DATA_DIR / str(rel).replace("\\", "/")
assert image_path(test_df["path"].iloc[0]).exists()
base_probs = np.load(ADAPTER_DIR / "test_probs.npy")
assert base_probs.shape == (len(test_df), 4)
print("test", len(test_df), "| 수혁 640px 확률", base_probs.shape)

# 5. 모델 불러오기 (4bit + 수혁 어댑터)

In [ ]:
try:
    model, _ = FastVisionModel.from_pretrained(str(ADAPTER_DIR), load_in_4bit=CONFIG["LOAD_IN_4BIT"], max_seq_length=2048)
except Exception as e:
    print("어댑터 폴더 직접 로드 실패 -> 기본 모델 + 어댑터 방식:", type(e).__name__)
    from peft import PeftModel
    model, _ = FastVisionModel.from_pretrained(CONFIG["BASE_REPO"], load_in_4bit=CONFIG["LOAD_IN_4BIT"], max_seq_length=2048)
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
FastVisionModel.for_inference(model)

# 해상도는 load_image 가 정하므로, processor 는 넓은 범위를 허용 (32 배수로 맞춘 이미지는 그대로 통과)
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"], min_pixels=256 * 256, max_pixels=1024 * 1024)
except TypeError:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"])
tok = processor.tokenizer
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f}GB")

# 6. 채점 함수 · 재현 확인 (93% 이상 일치해야 진행)

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)
def build_mc_prompt(question, opts):
    return (f"{question}\n(a) {opts[0]}\n(b) {opts[1]}\n(c) {opts[2]}\n(d) {opts[3]}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

LETTERS = list("abcd")
LETTER_IDS = []
for ch in LETTERS:
    ids = tok.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, ch
    LETTER_IDS.append(ids[0])

def load_image(rel, side):
    pix = side * side
    im = Image.open(image_path(rel))
    try:
        im.draft("RGB", (side * 2, side * 2))
    except Exception:
        pass
    im = im.convert("RGB")
    w, h = im.size
    s = (pix / (w * h)) ** 0.5
    return im.resize((max(32, round(w * s / 32) * 32), max(32, round(h * s / 32) * 32)), Image.BICUBIC)

@torch.no_grad()
def predict_probs(df, side, shift, desc):
    """반환: (N, 4) — 항상 '원래' a/b/c/d 순서의 확률"""
    out = np.zeros((len(df), 4), dtype=np.float32)
    order = [(k + shift) % 4 for k in range(4)]          # 화면 k 번째 칸에 원래 몇 번 보기를 보여줄지
    for i in tqdm(range(len(df)), desc=desc, unit="it"):
        row = df.iloc[i]
        img = load_image(row["path"], side)
        orig = [row["a"], row["b"], row["c"], row["d"]]
        msgs = [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text", "text": build_mc_prompt(row["question"], [orig[j] for j in order])}]},
        ]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inputs).logits[:, -1, :].float()
        p = torch.softmax(logits[0, LETTER_IDS], dim=-1).cpu().numpy()
        for k, j in enumerate(order):
            out[i, j] = p[k]
    return out

# --- 재현 확인 : 불러온 모델이 학습 직후와 같은 답을 내는지 (test 100문항) ---
t0 = time.time()
chk = predict_probs(test_df.head(100), 640, 0, "재현 확인 (640px · 수혁 PC 결과와 비교)")
sec640 = (time.time() - t0) / 100
agree = float(np.mean(chk.argmax(1) == base_probs[:100].argmax(1)))
print(f"학습 직후 답과 일치율 {agree:.0%} | 640px 문항당 {sec640:.2f}초")
assert agree >= 0.93, "불러온 모델이 수혁 님 PC 결과와 다르게 답합니다 (BASE_REPO / LOAD_IN_4BIT 확인)"

# 7. 속도 측정 · 예산 안에서 실행할 조합 선택

In [ ]:
# 해상도별 속도 측정 -> 예산 안에서 실행할 조합 선택
speed = {640: sec640}
for side in sorted({s for s, _ in CONFIG["PASSES"]} - {640}):
    t0 = time.time(); predict_probs(test_df.head(15), side, 0, f"속도 측정 {side}px")
    speed[side] = (time.time() - t0) / 15
N = len(test_df)
plan, used = [], 0.0
for side, shift in CONFIG["PASSES"]:
    if (TTA_DIR / f"tta_px{side}_s{shift}.npy").exists():
        plan.append((side, shift, 0.0)); continue          # 이미 끝난 조합 (세션 재시작 시 건너뜀)
    h = speed[side] * N / 3600
    if used + h <= CONFIG["BUDGET_HOURS"]:
        plan.append((side, shift, h)); used += h
print("문항당 초:", {k: round(v, 2) for k, v in speed.items()})
for side, shift, h in plan:
    print(f"  실행 예정: {side}px · 보기 {shift}칸 이동 · {'이미 있음' if h == 0 else f'{h * 60:.0f}분'}")
print(f"예상 합계 {used:.1f}시간 ≈ {used * 15:.0f}유닛")
assert plan, "예산 안에 드는 조합이 없습니다. BUDGET_HOURS 를 늘리세요"

# 8. TTA 실행 (조합마다 드라이브 저장)

In [ ]:
for side, shift, h in plan:
    f = TTA_DIR / f"tta_px{side}_s{shift}.npy"
    if f.exists():
        continue
    probs = predict_probs(test_df, side, shift, f"TTA {side}px · 이동{shift}")
    np.save(f, probs)
    agree = float(np.mean(probs.argmax(1) == base_probs.argmax(1)))
    print(f"  저장 {f.name} | 원래 640px 답과 일치율 {agree:.4f}")

# 9. 수혁 TTA 평균 · 단독 제출 파일

In [ ]:
L = np.array(LETTERS)
parts = {(640, 0): base_probs}
for f in sorted(TTA_DIR.glob("tta_px*_s*.npy")):
    side, shift = int(f.stem.split("_px")[1].split("_s")[0]), int(f.stem.split("_s")[-1])
    parts[(side, shift)] = np.load(f)
print("모인 조합:", sorted(parts))

def write(tag, keys):
    keys = [k for k in keys if k in parts]
    if len(keys) < 2:
        return
    avg = np.mean([parts[k] for k in keys], axis=0)
    ans = L[avg.argmax(1)]
    sub = sample_sub.copy()
    sub[sample_sub.columns[1]] = sub[sample_sub.columns[0]].map(dict(zip(test_df[sample_sub.columns[0]], ans)))
    assert sub[sample_sub.columns[1]].isna().sum() == 0
    path = SUB_DIR / f"submission_수혁_{tag}.csv"
    sub.to_csv(path, index=False)
    np.save(TTA_DIR / f"{tag}.npy", avg)                      # 앙상블 노트북에 넣을 확률
    changed = int(np.sum(ans != L[base_probs.argmax(1)]))
    print(f"[{tag}] 조합 {keys} | 원래와 달라진 답 {changed}개 ({changed / len(ans):.1%}) | {path.name}")

write("tta_all",  list(parts))
write("tta_res",  [k for k in parts if k[1] == 0])
write("tta_perm", [k for k in parts if k[0] == 640])
print("\n캐글에 위 파일들을 각각 제출해서 0.9407 과 비교하세요. 설명란 예: tta_all | 0.9407 모델 + TTA")

# 10. 동현 TTA 와 앙상블 · 제출 파일

In [ ]:
# 동현 TTA(0.9568) · 수혁 B3(1024px) 를 드라이브에서 찾아 수혁 TTA 와 섞기
NFC = lambda s: unicodedata.normalize("NFC", s)   # 7번 셀의 N(문항 수)과 이름 충돌 방지
L = np.array(list("abcd"))
def find(tag):
    hit = [p for p in MY.rglob("*") if p.is_file() and tag in NFC(p.name) and "ensemble_outputs" not in str(p)]
    return hit[0] if hit else None

su = np.load(TTA_DIR / "tta_all.npy") if (TTA_DIR / "tta_all.npy").exists() else base_probs
dh_f, b3_f = find("동현_TTA_8B_640px"), find("수혁_B3_3B_px1024")
print("동현 TTA:", dh_f, "\n수혁 B3 :", b3_f)

if dh_f is not None:
    np.save(dh_f.parent / "수혁_TTA_8B_640px.npy", su)          # 앙상블 폴더에 수혁 TTA 도 보관
    dh = np.load(dh_f); dh = dh / dh.sum(1, keepdims=True)
    parts = {"동현TTA": dh, "수혁TTA": su}
    if b3_f is not None:
        b3 = np.load(b3_f); parts["B3"] = b3 / b3.sum(1, keepdims=True)
    agree = float(np.mean(dh.argmax(1) == su.argmax(1)))
    print(f"동현 TTA vs 수혁 TTA 답 일치율 {agree:.4f}  (0.95 안팎이면 정상 · 낮을수록 섞을 가치 큼)")

    def save(tag, w):
        ks = [k for k in w if k in parts]
        ww = np.array([w[k] for k in ks]); ww = ww / ww.sum()
        avg = sum(wi * parts[k] for wi, k in zip(ww, ks))
        ans = L[avg.argmax(1)]
        sub = sample_sub.copy()
        sub[sample_sub.columns[1]] = sub[sample_sub.columns[0]].map(dict(zip(test_df[sample_sub.columns[0]], ans)))
        assert sub[sample_sub.columns[1]].isna().sum() == 0
        sub.to_csv(SUB_DIR / f"submission_{tag}.csv", index=False)
        print(f"[{tag}] 동현 TTA 와 다른 답 {int(np.sum(ans != L[dh.argmax(1)]))}개 | " +
              ", ".join(f"{k} {wi:.2f}" for k, wi in zip(ks, ww)))

    save("tta2_equal",   {"동현TTA": 0.5, "수혁TTA": 0.5})
    save("tta2_dh60",    {"동현TTA": 0.6, "수혁TTA": 0.4})
    save("tta2_b3",      {"동현TTA": 0.4, "수혁TTA": 0.4, "B3": 0.2})
    print("\n저장:", SUB_DIR)
else:
    print("동현 TTA 파일을 못 찾아 앙상블은 건너뜀 (수혁 TTA 단독 제출 파일은 위 셀에서 생성됨)")